In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
df = pd.read_csv("../data/merged_dataset_organized_47photic.csv")


# --------------------------------------------------
# 2. Define baseline and upper depth
# --------------------------------------------------
BASELINE_DEPTH = 2
UPPER_DEPTH = 20  # median Secchi depth

# --------------------------------------------------
# 3. Create summed variables (2m to 20m per cast)
# --------------------------------------------------
df_range = df[(df["DEPTH"] >= BASELINE_DEPTH) & 
              (df["DEPTH"] <= UPPER_DEPTH)]

summed = df_range.groupby("CAST_COUNT").agg(
    XMISS_SUMMED=("XMISS", "sum"),
    CHL_A_SUMMED=("CHL_A", "sum")
).reset_index()

# --------------------------------------------------
# 4. Keep only photic_zone == TRUE rows (one row per cast)
# --------------------------------------------------
df_photic = df[df["PHOTIC_ZONE"] == True].copy()

# --------------------------------------------------
# 5. Merge summed variables onto photic-zone dataframe (each cast will have one row)
# --------------------------------------------------
df_final = df_photic.merge(summed, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 6. Impute ESTCHL_STACORR (median imputation)
# --------------------------------------------------
median_estchl = df_final["ESTCHL_STACORR"].median()
df_final["ESTCHL_STACORR"] = df_final["ESTCHL_STACORR"].fillna(median_estchl)

# --------------------------------------------------
# 7A. Two-point Beer-Lambert K_PAR (original formula)
# --------------------------------------------------

# Baseline PAR at 2m
baseline_par = df[df["DEPTH"] == BASELINE_DEPTH][
    ["CAST_COUNT", "PAR"]
].rename(columns={"PAR": "PAR_BASELINE"}).drop_duplicates("CAST_COUNT")

df_final = df_final.merge(baseline_par, on="CAST_COUNT", how="left")

df_final["K_PAR"] = -np.log(
    df_final["PAR"] / df_final["PAR_BASELINE"]
) / df_final["DEPTH"]

df_final = df_final.drop(columns=["PAR_BASELINE"])  # remove intermediate column


# --------------------------------------------------
# 7B. Regression slope-based K_PAR_SLOPE
# --------------------------------------------------

kpar_slope_results = []

for cast_id, cast_df in df.groupby("CAST_COUNT"):

    # Get photic depth
    photic_row = cast_df[cast_df["PHOTIC_ZONE"] == True]
    if photic_row.empty:
        continue

    photic_depth = photic_row["DEPTH"].values[0]

    # Subset depths between 2m and photic depth
    subset = cast_df[
        (cast_df["DEPTH"] >= BASELINE_DEPTH) &
        (cast_df["DEPTH"] <= photic_depth)
    ].copy()

    subset = subset[subset["PAR"] > 0]

    if len(subset) < 2:
        continue

    subset["LOG_PAR"] = np.log(subset["PAR"])

    X = subset[["DEPTH"]].values
    y = subset["LOG_PAR"].values

    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]

    kpar_slope_results.append({
        "CAST_COUNT": cast_id,
        "K_PAR_SLOPE": -slope
    })

kpar_slope_df = pd.DataFrame(kpar_slope_results)

df_final = df_final.merge(kpar_slope_df, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 8. Save final dataset
# --------------------------------------------------
df_final.to_parquet("../data/Parquet/enhanced_summed_47photic.parquet", index=False)

print(df_final.head())

   ORD_OCC    CAST_ID         DATE_TIME_UTC         DATE_TIME_PST  LAT_DEC  \
0      1.0  9308_001d  1993-08-11T12:06:56Z  1993-08-11T04:06:56Z    -99.0   
1      6.0  9308_006d  1993-08-12T10:57:24Z  1993-08-12T02:57:24Z    -99.0   
2     11.0  9308_011d  1993-08-13T14:53:14Z  1993-08-13T06:53:14Z    -99.0   
3     14.0  9308_014d  1993-08-14T10:19:40Z  1993-08-14T02:19:40Z    -99.0   
4     18.0  9308_018d  1993-08-15T10:12:16Z  1993-08-15T02:12:16Z    -99.0   

   LON_DEC       STA_ID  LINE    STA  DEPTH  ...  Secchi  IntChl  IntC14  \
0    -99.0  093.3 026.7  93.3   26.7   29.0  ...     NaN     NaN     NaN   
1    -99.0  093.3 045.0  93.3   45.0   47.0  ...     NaN     NaN     NaN   
2    -99.0  093.3 080.0  93.3   80.0   59.0  ...     NaN     NaN     NaN   
3    -99.0  093.3 110.0  93.3  110.0   77.0  ...     NaN     NaN     NaN   
4    -99.0  090.0 100.0  90.0  100.0   58.0  ...     NaN     NaN     NaN   

   TimeZone  Visibility  PHOTIC_ZONE  XMISS_SUMMED  CHL_A_SUMMED     K_PAR

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/Parquet/enhanced_summed_47photic.parquet")

print("=== SHAPE ===")
print(df.shape)  # rows should = number of unique casts

print("\n=== DUPLICATES ===")
print("Duplicate CAST_COUNT rows:", df["CAST_COUNT"].duplicated().sum())  # should be 0

print("\n=== NEW COLUMNS ===")
for col in ["XMISS_SUMMED", "CHL_A_SUMMED", "K_PAR", "K_PAR_SLOPE", "ESTCHL_STACORR"]:
    print(f"\n{col}:")
    print(df[col].describe().round(4))
    print(f"  NaNs: {df[col].isna().sum()}")

print("\n=== SANITY CHECKS ===")
# K_PAR should be positive (light attenuates with depth)
print("K_PAR negative values:", (df["K_PAR"] < 0).sum())  # should be 0 or very few
print("K_PAR_SLOPE negative values:", (df["K_PAR_SLOPE"] < 0).sum())  # should be 0 or very few

# XMISS_SUMMED should be > 0
print("XMISS_SUMMED zeros:", (df["XMISS_SUMMED"] == 0).sum())

# ESTCHL_STACORR should have no NaNs after imputation
print("ESTCHL_STACORR NaNs:", df["ESTCHL_STACORR"].isna().sum())  # should be 0

# PAR_BASELINE should all be at 2m — check it's reasonable
print("\nPAR_BASELINE range:", df["PAR_BASELINE"].min(), "to", df["PAR_BASELINE"].max())

=== SHAPE ===
(1971, 42)

=== DUPLICATES ===
Duplicate CAST_COUNT rows: 0

=== NEW COLUMNS ===

XMISS_SUMMED:
count    1971.0000
mean     1330.1991
std       619.3403
min     -1881.0000
25%      1451.6750
50%      1579.8600
75%      1629.7650
max      3246.0800
Name: XMISS_SUMMED, dtype: float64
  NaNs: 0

CHL_A_SUMMED:
count    1971.0000
mean        3.4674
std         7.7274
min         0.0000
25%         0.3600
50%         1.0000
75%         2.8500
max       108.1000
Name: CHL_A_SUMMED, dtype: float64
  NaNs: 0

K_PAR:
count    1971.0000
mean        0.1285
std         0.3368
min         0.0224
25%         0.0600
50%         0.0855
75%         0.1287
max         7.3742
Name: K_PAR, dtype: float64
  NaNs: 0

K_PAR_SLOPE:
count    1971.0000
mean        0.1516
std         0.6460
min         0.0126
25%         0.0528
50%         0.0811
75%         0.1328
max        14.7483
Name: K_PAR_SLOPE, dtype: float64
  NaNs: 0

ESTCHL_STACORR:
count    1971.0000
mean        1.2972
std         2.6896